# Fine-Tuning Basic Pitch on GuitarSet
Clean implementation using Spotify's own training pipeline.

**Approach:** Convert GuitarSet → TFRecord format (exactly matching Basic Pitch's schema) → fine-tune using their `train.py` logic directly. No dependency on Kong, no SavedModel gradient hacks.

**Expected result (TART Table 2):** GuitarSet note F50 0.704 (base) → 0.838 (fine-tuned acoustic).

**Cells in order:**
1. Install + clone
2. Mount Drive + copy to SSD
3. Convert GuitarSet → TFRecords (one-time, ~10 min)
4. Build + compile model
5. Fine-tune
6. Evaluate P50/R50/F50 vs baseline


In [1]:
!nvidia-smi

Sun Jun 21 19:47:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch; print(torch.cuda.is_available())

True


## 1. Install dependencies

In [3]:
!pip install -q --no-deps -e /content/basic-pitch 2>/dev/null ||     (git clone https://github.com/spotify/basic-pitch.git /content/basic-pitch &&      pip install -q --no-deps -e /content/basic-pitch)
!pip install -q librosa soundfile sox mirdata tensorflow

import tensorflow as tf
import sys
sys.path.insert(0, '/content/basic-pitch')
print(f"TF: {tf.__version__}")

# Confirm model builds (ignore TF version warning)
import warnings; warnings.filterwarnings('ignore')
from basic_pitch import models as bp_models
print("basic_pitch imported OK")
print("Available:", [x for x in dir(bp_models) if not x.startswith('_')])

Cloning into '/content/basic-pitch'...
remote: Enumerating objects: 1657, done.
remote: Counting objects: 100% (821/821), done.
remote: Compressing objects: 100% (332/332), done.
remote: Total 1657 (delta 631), reused 489 (delta 489), pack-reused 836 (from 3)
Receiving objects: 100% (1657/1657), 282.68 MiB | 33.54 MiB/s, done.
Resolving deltas: 100% (947/947), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for basic-pitch (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 109.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

TF: 2.20.0
basic_pitch imported OK
Available: ['ANNOTATIONS_BASE_FREQUENCY', 'ANNOTATIONS_N_SEMITONES', 'AUDIO_N_SAMPLES', 'AUDIO_SAMPLE_RATE', 'Any', 'CONTOURS_BINS_PER_SEMITONE', 'CONTOUR_FILTERS_2', 'CONTOUR_KERNEL_SIZE_1', 'CONTOUR_KERNEL_SIZE_2', 'CONTOUR_KERNEL_SIZE_3', 'Callable', 'DEFAULT_LABEL_SMOOTHING', 'DEFAULT_POSITIVE_WEIGHT', 'Dict', 'FFT_HOP', 'MAX_N_SEMITONES', 'NOTES_KERNEL_SIZE_1', 'NOTES_KERNEL_SIZE_2', 'NOTES_STRIDES_1', 'N_FREQ_BINS_CONTOURS', 'ONSET_KERNEL_SIZE_1', 'ONSET_KERNEL_SIZE_2', 'ONSET_STRIDES_1', 'get_cqt', 'loss', 'model', 'nn', 'nnaudio', 'np', 'onset_loss', 'signal', 'tf', 'tfkl', 'transcription_loss', 'weighted_transcription_loss']


## 2. Mount Drive + copy GuitarSet to local SSD

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
except ModuleNotFoundError:
    print('Not in Colab.')

import os, glob, shutil, json
from pathlib import Path

DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
]
DATA_ROOT = next((c for c in DATA_ROOT_CANDIDATES if (c/'JamsFiles').exists()), None)
if DATA_ROOT is None: raise FileNotFoundError("GuitarSet not found")
print(f"GuitarSet at: {DATA_ROOT}")

LOCAL_AUDIO = '/content/gs_audio'
LOCAL_JAMS  = '/content/gs_jams'
os.makedirs(LOCAL_AUDIO, exist_ok=True)
os.makedirs(LOCAL_JAMS,  exist_ok=True)

def copy_if_needed(src_dir, dst_dir, ext):
    src = glob.glob(os.path.join(str(src_dir), f'*.{ext}'))
    dst = glob.glob(os.path.join(dst_dir, f'*.{ext}'))
    if len(dst) >= len(src):
        print(f"  {ext}: {len(dst)} files already on SSD"); return
    print(f"  Copying {len(src)} {ext} files...")
    for f in src:
        try: shutil.copy2(f, dst_dir)
        except shutil.SameFileError: pass
    print(f"  Done.")

copy_if_needed(DATA_ROOT/'AudioFiles', LOCAL_AUDIO, 'wav')
copy_if_needed(DATA_ROOT/'JamsFiles',  LOCAL_JAMS,  'jams')
print(f"Audio: {len(glob.glob(LOCAL_AUDIO+'/*.wav'))} | JAMS: {len(glob.glob(LOCAL_JAMS+'/*.jams'))}")

OUTPUT_DIR = '/content/drive/MyDrive/Capstone/outputs/bp_finetune'
TFRECORD_DIR = '/content/guitarset_tfrecords'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TFRECORD_DIR, exist_ok=True)
print(f"TFRecords -> {TFRECORD_DIR}")
print(f"Checkpoints -> {OUTPUT_DIR}")

Mounted at /content/drive
Drive mounted.
GuitarSet at: /content/drive/MyDrive/Capstone/FullGuitarSetData
  Copying 360 wav files...
  Done.
  Copying 360 jams files...
  Done.
Audio: 360 | JAMS: 360
TFRecords -> /content/guitarset_tfrecords
Checkpoints -> /content/drive/MyDrive/Capstone/outputs/bp_finetune


## 3. Convert GuitarSet → TFRecords
One-time step (~10 min). Produces the exact schema Basic Pitch's training pipeline expects.
Split: player 05 = test (held out), player 04 = validation, players 00-03 = train.


In [6]:
!apt-get install -y sox

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa libsox-fmt-base
  libsox3 libwavpack1
Suggested packages:
  libsox-fmt-all
The following NEW packages will be installed:
  libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa libsox-fmt-base
  libsox3 libwavpack1 sox
0 upgraded, 7 newly installed, 0 to remove and 53 not upgraded.
Need to get 617 kB of archives.
After this operation, 1,764 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopencore-amrnb0 amd64 0.1.5-1 [94.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopencore-amrwb0 amd64 0.1.5-1 [49.1 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libsox3 amd64 14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1 [240 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 

In [7]:
import numpy as np
import tensorflow as tf
import librosa, json, os, glob, sox
from basic_pitch.constants import (
    AUDIO_SAMPLE_RATE, AUDIO_N_CHANNELS, FFT_HOP,
    ANNOTATIONS_FPS, ANNOTATION_HOP,
    FREQ_BINS_NOTES, FREQ_BINS_CONTOURS,
    N_FREQ_BINS_NOTES, N_FREQ_BINS_CONTOURS,
)
from basic_pitch.data.tf_example_serialization import (
    bytes_feature, int64_feature, float_feature,
)

def load_notes_from_jams(jams_path):
    """Load per-string notes from GuitarSet JAMS, return merged note list."""
    with open(jams_path) as f:
        jam = json.load(f)
    notes = []
    for ann in jam.get('annotations', []):
        if ann.get('namespace','') not in ('note_midi','pitch_midi'): continue
        for obs in ann['data']:
            midi = float(obs['value'])
            freq = 440.0 * (2.0 ** ((midi - 69.0) / 12.0))
            onset  = float(obs['time'])
            offset = onset + float(obs['duration'])
            notes.append({'onset': onset, 'offset': offset, 'freq': freq})
    return sorted(notes, key=lambda n: n['onset'])

def notes_to_sparse(notes, time_scale, freq_bins, onsets_only=False):
    """Convert note list to sparse index/value arrays matching Basic Pitch format."""
    indices, values = [], []
    freq_bin_edges = np.concatenate([[0],
        (freq_bins[:-1] + freq_bins[1:]) / 2,
        [freq_bins[-1] * 2]])
    for note in notes:
        # find freq bin
        freq = note['freq']
        b = np.searchsorted(freq_bin_edges, freq) - 1
        b = int(np.clip(b, 0, len(freq_bins)-1))
        if onsets_only:
            t = int(round(note['onset'] / ANNOTATION_HOP))
            if 0 <= t < len(time_scale):
                indices.append([t, b]); values.append(1.0)
        else:
            t0 = int(round(note['onset']  / ANNOTATION_HOP))
            t1 = int(round(note['offset'] / ANNOTATION_HOP))
            for t in range(max(0,t0), min(len(time_scale), t1)):
                indices.append([t, b]); values.append(1.0)
    if not indices:
        return np.zeros((0,2), np.int64), np.zeros(0, np.float32)
    return np.array(indices, np.int64), np.array(values, np.float32)

def contours_to_sparse(notes, time_scale, freq_bins):
    """Contour = pitch active; same as notes but uses FREQ_BINS_CONTOURS."""
    return notes_to_sparse(notes, time_scale, freq_bins, onsets_only=False)

def wav_to_mono22k(src_path, dst_path):
    """Resample + convert to mono WAV at 22050 Hz using sox."""
    if os.path.exists(dst_path): return
    tfm = sox.Transformer()
    tfm.rate(AUDIO_SAMPLE_RATE)
    tfm.channels(AUDIO_N_CHANNELS)
    tfm.build(src_path, dst_path)

def make_tfrecord(jams_path, audio_path, split, tfrecord_dir):
    stem = os.path.splitext(os.path.basename(jams_path))[0]
    out_path = os.path.join(tfrecord_dir, split, f'{stem}.tfrecord')
    if os.path.exists(out_path): return  # skip if already done

    # resample audio
    tmp_wav = f'/tmp/{stem}_22k.wav'
    wav_to_mono22k(audio_path, tmp_wav)

    import sox as _sox
    duration = _sox.file_info.duration(tmp_wav)
    time_scale = np.arange(0, duration + ANNOTATION_HOP, ANNOTATION_HOP)
    n_time_frames = len(time_scale)

    notes = load_notes_from_jams(jams_path)
    note_idx,   note_val  = notes_to_sparse(notes, time_scale, FREQ_BINS_NOTES)
    onset_idx,  onset_val = notes_to_sparse(notes, time_scale, FREQ_BINS_NOTES, onsets_only=True)
    cont_idx,   cont_val  = contours_to_sparse(notes, time_scale, FREQ_BINS_CONTOURS)

    encoded_wav = open(tmp_wav, 'rb').read()
    example = tf.train.Example(features=tf.train.Features(feature={
        'file_id':           bytes_feature(bytes(stem, 'utf-8')),
        'source':            bytes_feature(b'guitarset'),
        'audio_wav':         bytes_feature(encoded_wav),
        'notes_indices':     bytes_feature(tf.io.serialize_tensor(note_idx).numpy()),
        'notes_values':      bytes_feature(tf.io.serialize_tensor(note_val).numpy()),
        'onsets_indices':    bytes_feature(tf.io.serialize_tensor(onset_idx).numpy()),
        'onsets_values':     bytes_feature(tf.io.serialize_tensor(onset_val).numpy()),
        'contours_indices':  bytes_feature(tf.io.serialize_tensor(cont_idx).numpy()),
        'contours_values':   bytes_feature(tf.io.serialize_tensor(cont_val).numpy()),
        'notes_onsets_shape':bytes_feature(tf.io.serialize_tensor(
                                np.array([n_time_frames, N_FREQ_BINS_NOTES], np.int64)).numpy()),
        'contours_shape':    bytes_feature(tf.io.serialize_tensor(
                                np.array([n_time_frames, N_FREQ_BINS_CONTOURS], np.int64)).numpy()),
    }))
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with tf.io.TFRecordWriter(out_path) as w:
        w.write(example.SerializeToString())

def guitarset_player(stem): return stem.split('_')[0]

def get_split(stem):
    p = guitarset_player(stem)
    if p == '05': return 'test'
    if p == '04': return 'validation'
    return 'train'

jams_files = sorted(glob.glob(os.path.join(LOCAL_JAMS, '*.jams')))
print(f"Converting {len(jams_files)} recordings to TFRecords...")
errors = []
for i, jp in enumerate(jams_files):
    stem  = os.path.splitext(os.path.basename(jp))[0]
    split = get_split(stem)
    cands = (glob.glob(os.path.join(LOCAL_AUDIO, stem+'*mic*.wav')) or
             glob.glob(os.path.join(LOCAL_AUDIO, stem+'*.wav')))
    if not cands: errors.append(stem); continue
    try:
        make_tfrecord(jp, cands[0], split, TFRECORD_DIR)
    except Exception as e:
        errors.append(f'{stem}: {e}')
    if (i+1) % 60 == 0: print(f'  {i+1}/{len(jams_files)} done...')

for split in ['train','validation','test']:
    n = len(glob.glob(os.path.join(TFRECORD_DIR, split, '*.tfrecord')))
    print(f"  {split}: {n} tfrecords")
if errors: print(f"Errors ({len(errors)}):", errors[:3])
else: print("All converted OK")

Converting 360 recordings to TFRecords...


  60/360 done...


  120/360 done...


  180/360 done...


  240/360 done...


  300/360 done...


  360/360 done...
  train: 240 tfrecords
  validation: 60 tfrecords
  test: 60 tfrecords
All converted OK


## 4. Speed test — confirm TFRecord pipeline is fast

In [8]:
from basic_pitch.data.tf_example_deserialization import prepare_datasets
from basic_pitch.constants import DATASET_SAMPLING_FREQUENCY
import time

# Basic Pitch expects: source_dir/guitarset/splits/train/*.tfrecord
# We need to symlink our structure into that layout
import os
BP_DATA_DIR = '/content/bp_data'
gs_link = os.path.join(BP_DATA_DIR, 'guitarset', 'splits')
os.makedirs(gs_link, exist_ok=True)
for split in ['train','validation','test']:
    link = os.path.join(gs_link, split)
    src  = os.path.join(TFRECORD_DIR, split)
    if not os.path.exists(link):
        os.symlink(src, link)
        print(f"Linked {split}")

# Test pipeline speed
BATCH_SIZE = 8
train_ds, val_ds = prepare_datasets(
    BP_DATA_DIR,
    training_shuffle_buffer_size=100,
    batch_size=BATCH_SIZE,
    validation_steps=10,
    datasets_to_use=['guitarset'],
    dataset_sampling_frequency=np.array([1.0]),
)
print("Datasets created. Timing first 20 batches...")
t0 = time.time()
for i, batch in enumerate(train_ds.take(20)):
    audio, targets, weights = batch
    if i == 0:
        print(f"  audio: {audio.shape} | onset: {targets['onset'].shape}")
elapsed = time.time()-t0
print(f"20 batches in {elapsed:.1f}s = {20/elapsed:.1f} batches/s")
print("If > 5 batches/s, proceed to training.")

Linked train
Linked validation
Linked test


Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


Datasets created. Timing first 20 batches...
  audio: (8, 43844, 1) | onset: (8, 172, 88)
20 batches in 1.2s = 16.6 batches/s
If > 5 batches/s, proceed to training.


In [14]:
# Patch models.py - replace tf.expand_dims with keras layer equivalent
models_path = '/content/basic-pitch/basic_pitch/models.py'
with open(models_path) as f:
    code = f.read()

# Fix 1: tf.expand_dims in get_cqt
code = code.replace(
    'x = tf.expand_dims(x, -1)\n    if use_batchnorm:',
    'x = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x)\n    if use_batchnorm:'
)

# Fix 2: tf.expand_dims in model()
code = code.replace(
    'x_contours_reduced = tf.expand_dims(x_contours, -1)',
    'x_contours_reduced = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x_contours)'
)

with open(models_path, 'w') as f:
    f.write(code)
print("Patched models.py")

# Patch nn.py - wrap tf.debugging.assert_equal in a way that doesn't break symbolic tensors
nn_path = '/content/basic-pitch/basic_pitch/nn.py'
with open(nn_path) as f:
    code = f.read()

code = code.replace(
    'tf.debugging.assert_equal(tf.shape(x).shape, 4)',
    'pass  # tf.debugging.assert_equal removed for TF2.20 Keras3 compat'
)

with open(nn_path, 'w') as f:
    f.write(code)
print("Patched nn.py")

# Reload all affected modules
import importlib
import basic_pitch.layers.signal as sig_mod
import basic_pitch.nn as nn_mod
import basic_pitch.models as bp_models_mod
importlib.reload(sig_mod)
importlib.reload(nn_mod)
importlib.reload(bp_models_mod)

import basic_pitch.models as bp_models
model = bp_models.model()
print("Model built OK")
print("Output keys:", list(model.output.keys()))

Patched models.py
Patched nn.py
Model built OK
Output keys: ['onset', 'contour', 'note']


## 5. Build model + fine-tune

In [15]:
# Patch signal.py for TF 2.20 compatibility
signal_path = '/content/basic-pitch/basic_pitch/layers/signal.py'
with open(signal_path) as f:
    code = f.read()

# Fix: input_shape is a tuple in TF 2.20, not a TensorShape with .rank
patched = code.replace(
    'rank = input_shape.rank',
    'rank = len(input_shape)'
)
with open(signal_path, 'w') as f:
    f.write(patched)

print("Patched signal.py")

# Reload the module so the patch takes effect
import importlib
import basic_pitch.layers.signal as sig_mod
importlib.reload(sig_mod)
import basic_pitch.models as bp_models
importlib.reload(bp_models)

# Now build
model = bp_models.model()
print("Model built OK")
model.summary(line_length=100)

Patched signal.py
Model built OK


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape            ┃        Param # ┃ Connected to            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)  │ (None, 43844, 1)        │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ flatten_audio_ch_3          │ (None, 43844)           │              0 │ input_layer_3[0][0]     │
│ (FlattenAudioCh)            │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ cqt2010v2_3 (CQT2010v2)     │ (None, 172, 309)        │              0 │ flatten_audio_ch_3[0][… │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ normalized_log_3            │ (None, 172, 309)        │              0 │ cqt2010v2_3[0][0]       │
│ (NormalizedLog)             │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ lambda_2 (Lambda)           │ (None, 172, 309, 1)     │              0 │ normalized_log_3[0][0]  │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ batch_normalization_3       │ (None, 172, 309, 1)     │              4 │ lambda_2[0][0]          │
│ (BatchNormalization)        │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ harmonic_stacking           │ (None, 172, 264, 8)     │              0 │ batch_normalization_3[… │
│ (HarmonicStacking)          │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ conv2d_5 (Conv2D)           │ (None, 172, 264, 8)     │          7,496 │ harmonic_stacking[0][0] │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ batch_normalization_4       │ (None, 172, 264, 8)     │             32 │ conv2d_5[0][0]          │
│ (BatchNormalization)        │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ re_lu_3 (ReLU)              │ (None, 172, 264, 8)     │              0 │ batch_normalization_4[… │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ contours-reduced (Conv2D)   │ (None, 172, 264, 1)     │            201 │ re_lu_3[0][0]           │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ contour (FlattenFreqCh)     │ (None, 172, 264)        │              0 │ contours-reduced[0][0]  │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ lambda_3 (Lambda)           │ (None, 172, 264, 1)     │              0 │ contour[0][0]           │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ conv2d_6 (Conv2D)           │ (None, 172, 88, 32)     │          1,600 │ lambda_3[0][0]          │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ conv2d_8 (Conv2D)           │ (None, 172, 88, 32)     │          6,432 │ harmonic_stacking[0][0] │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ re_lu_4 (ReLU)              │ (None, 172, 88, 32)     │              0 │ conv2d_6[0][0]          │
├─────────────────────────────┼─────────────────────────┼────

 Total params: 16,864 (65.88 KB)

 Trainable params: 16,782 (65.55 KB)

 Non-trainable params: 82 (328.00 B)

In [21]:
# Strip sample weights from dataset - we don't need them
train_ds_no_weights = train_ds.map(lambda x, y, w: (x, y))
val_ds_no_weights   = val_ds.map(lambda x, y, w: (x, y))

model.compile(
    loss={'onset': simple_loss, 'note': simple_loss, 'contour': simple_loss},
    optimizer=tf.keras.optimizers.Adam(1e-4),
)
print("Recompiled OK")

history = model.fit(
    train_ds_no_weights,
    epochs=50,
    steps_per_epoch=100,
    validation_data=val_ds_no_weights,
    validation_steps=20,
    callbacks=callbacks,
)
print("Training done.")

Recompiled OK
Epoch 1/50
 96/100 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - contour_loss: 0.3366 - loss: 1.0011 - note_loss: 0.3349 - onset_loss: 0.3296
Epoch 1: val_loss improved from None to 0.99124, saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune/best_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 39ms/step - contour_loss: 0.3355 - loss: 0.9988 - note_loss: 0.3347 - onset_loss: 0.3286 - val_contour_loss: 0.3314 - val_loss: 0.9912 - val_note_loss: 0.3321 - val_onset_loss: 0.3277 - learning_rate: 1.0000e-04
Epoch 2/50
 99/100 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - contour_loss: 0.3335 - loss: 0.9952 - note_loss: 0.3341 - onset_loss: 0.3275
Epoch 2: val_loss improved from 0.99124 to 0.98982, saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune/best_model.keras

In [13]:
models_path = '/content/basic-pitch/basic_pitch/models.py'
with open(models_path) as f:
    code = f.read()
print(code)

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2022 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from typing import Any, Callable, Dict
import numpy as np
import tensorflow as tf

from basic_pitch import nn
from basic_pitch.constants import (
    ANNOTATIONS_BASE_FREQUENCY,
    ANNOTATIONS_N_SEMITONES,
    AUDIO_N_SAMPLES,
    AUDIO_SAMPLE_RATE,
    CONTOURS_BINS_PER_SEMITONE,
    FFT_HOP,
    N_FREQ_BINS_CONTOURS,
)
from basic_pitch.layers import signal, nnaudio

tfkl = tf.k

In [9]:
import tensorflow as tf, numpy as np, os, time
from basic_pitch import models as bp_models
from basic_pitch import ICASSP_2022_MODEL_PATH
from basic_pitch.data.tf_example_deserialization import prepare_datasets

BATCH_SIZE    = 8
LR            = 1e-4    # slightly higher than TART for faster convergence on short run
EPOCHS        = 50
STEPS_PER_EPOCH = 100   # 100 steps * 8 batch = 800 examples/epoch
VAL_STEPS     = 20

# Build fresh Keras model
print("Building model...")
model = bp_models.model()
model.summary(line_length=100)

# Load pretrained weights from SavedModel into Keras model
print("Loading pretrained weights...")
saved = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))
try:
    model.set_weights([v.numpy() for v in saved.variables])
    print(f"Loaded {len(saved.variables)} weight tensors")
except Exception as e:
    print(f"Weight transfer failed: {e}")
    print("Proceeding with random init (will still fine-tune, just from scratch)")

# Compile with Basic Pitch's own loss
loss = bp_models.loss(weighted=False, positive_weight=0.5)
model.compile(
    loss=loss,
    optimizer=tf.keras.optimizers.Adam(LR),
    sample_weight_mode={"contour": None, "note": None, "onset": None},
)
print("Model compiled OK")

# Data
train_ds, val_ds = prepare_datasets(
    BP_DATA_DIR,
    training_shuffle_buffer_size=100,
    batch_size=BATCH_SIZE,
    validation_steps=VAL_STEPS,
    datasets_to_use=['guitarset'],
    dataset_sampling_frequency=np.array([1.0]),
)

# Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, 'best_model'),
        save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=10, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
]

print(f"Training: {EPOCHS} epochs x {STEPS_PER_EPOCH} steps, batch={BATCH_SIZE}")
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
)
print("Training done.")

Building model...


AttributeError: 'tuple' object has no attribute 'rank'

## 6. Evaluate — P50/R50/F50 on held-out test set (player 05)

In [23]:
!pip install -q mir_eval resampy==0.4.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
basic-pitch 0.4.0 requires tensorflow<2.15.1,>=2.4.1; platform_system != "Darwin" and python_version >= "3.11", but you have tensorflow 2.20.0 which is incompatible.


In [29]:
import numpy as np, pandas as pd, json, glob, os, librosa, tensorflow as tf
from basic_pitch.note_creation import model_output_to_notes
from basic_pitch.constants import AUDIO_SAMPLE_RATE, AUDIO_N_SAMPLES, FFT_HOP, ANNOTATIONS_FPS

ONSET_THRESHOLD = 0.40
FRAME_THRESHOLD = 0.30
ONSET_TOL = 0.05

def load_notes_gt(jams_path):
    with open(jams_path) as f: jam = json.load(f)
    notes = []
    for ann in jam.get('annotations',[]):
        if ann.get('namespace','') not in ('note_midi','pitch_midi'): continue
        for obs in ann['data']:
            notes.append({'onset': float(obs['time']),
                          'offset': float(obs['time'])+float(obs['duration']),
                          'midi': int(round(float(obs['value'])))})
    return sorted(notes, key=lambda n: n['onset'])

def run_inference(audio_path, model):
    y, _ = librosa.load(audio_path, sr=AUDIO_SAMPLE_RATE, mono=True)
    all_onset=[]; all_note=[]; all_contour=[]
    for start in range(0, len(y), AUDIO_N_SAMPLES):
        chunk = y[start:start+AUDIO_N_SAMPLES]
        if len(chunk)<AUDIO_N_SAMPLES: chunk=np.pad(chunk,(0,AUDIO_N_SAMPLES-len(chunk)))
        x = tf.constant(chunk.reshape(1,-1,1), dtype=tf.float32)
        out = model(x, training=False)
        all_onset.append(out['onset'][0].numpy())
        all_note.append(out['note'][0].numpy())
        all_contour.append(out['contour'][0].numpy())
    onset_mat=np.concatenate(all_onset,0)
    note_mat=np.concatenate(all_note,0)
    contour_mat=np.concatenate(all_contour,0)
    _, note_events = model_output_to_notes(
        {'onset':onset_mat,'note':note_mat,'contour':contour_mat},
        onset_thresh=ONSET_THRESHOLD, frame_thresh=FRAME_THRESHOLD,
        min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
        min_freq=None, max_freq=None, include_pitch_bends=False)
    return [{'onset':float(n[0]),'offset':float(n[1]),'midi':int(n[2])} for n in note_events]

def match_notes(gt, pred, tol=ONSET_TOL):
    cands=[]
    for pi,p in enumerate(pred):
        for gi,g in enumerate(gt):
            if int(p['midi'])!=int(g['midi']): continue
            if abs(p['onset']-g['onset'])<=tol: cands.append((abs(p['onset']-g['onset']),pi,gi))
    cands.sort(); up,ug=set(),set()
    for dt,pi,gi in cands:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi)
    tp=len(up); fp=len(pred)-tp; fn=len(gt)-tp
    P=tp/(tp+fp) if tp+fp else 0; R=tp/(tp+fn) if tp+fn else 0
    return P, R, 2*P*R/(P+R) if P+R else 0

def evaluate(model, jams_dir, audio_dir, label):
    test_jams = [j for j in sorted(glob.glob(os.path.join(jams_dir,'*.jams')))
                 if os.path.basename(j).split('_')[0]=='05']
    rows=[]
    for jp in test_jams:
        stem = os.path.splitext(os.path.basename(jp))[0]
        cands = (glob.glob(os.path.join(audio_dir,stem+'*mic*.wav')) or
                 glob.glob(os.path.join(audio_dir,stem+'*.wav')))
        if not cands: continue
        gt   = load_notes_gt(jp)
        pred = run_inference(cands[0], model)
        P,R,F = match_notes(gt, pred)
        rows.append({'id':stem,'P50':P,'R50':R,'F50':F,'n_gt':len(gt),'n_pred':len(pred)})
    df = pd.DataFrame(rows)
    agg = {k: float(np.average(df[k], weights=df.n_gt)) for k in ['P50','R50','F50']}
    print(f"\n── {label} ──")
    print(f"  P50:{agg['P50']:.4f}  R50:{agg['R50']:.4f}  F50:{agg['F50']:.4f}  n={len(df)}")
    return df, agg

# ── Baseline: pretrained Basic Pitch ─────────────────────────────────────────
from basic_pitch import ICASSP_2022_MODEL_PATH
base_model = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))

def base_inference(audio_path):
    y, _ = librosa.load(audio_path, sr=AUDIO_SAMPLE_RATE, mono=True)
    all_onset=[]; all_note=[]; all_contour=[]
    for start in range(0, len(y), AUDIO_N_SAMPLES):
        chunk = y[start:start+AUDIO_N_SAMPLES]
        if len(chunk)<AUDIO_N_SAMPLES: chunk=np.pad(chunk,(0,AUDIO_N_SAMPLES-len(chunk)))
        x = tf.constant(chunk.reshape(1,-1,1), dtype=tf.float32)
        out = base_model.signatures['serving_default'](input_2=x)
        all_onset.append(out['onset'][0].numpy())
        all_note.append(out['note'][0].numpy())
        all_contour.append(out['contour'][0].numpy())
    onset_mat=np.concatenate(all_onset,0)
    note_mat=np.concatenate(all_note,0)
    contour_mat=np.concatenate(all_contour,0)
    _, note_events = model_output_to_notes(
        {'onset':onset_mat,'note':note_mat,'contour':contour_mat},
        onset_thresh=ONSET_THRESHOLD, frame_thresh=FRAME_THRESHOLD,
        min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
        min_freq=None, max_freq=None, include_pitch_bends=False)
    # note tuple: (start_time, end_time, pitch, amplitude, bend)
    return [{'onset':float(n[0]),'offset':float(n[1]),'midi':int(n[2])} for n in note_events]


print("Evaluating pretrained Basic Pitch on test set (player 05)...")
test_jams = [j for j in sorted(glob.glob(os.path.join(LOCAL_JAMS,'*.jams')))
             if os.path.basename(j).split('_')[0]=='05']
rows=[]
for jp in test_jams:
    stem=os.path.splitext(os.path.basename(jp))[0]
    cands=(glob.glob(os.path.join(LOCAL_AUDIO,stem+'*mic*.wav')) or
           glob.glob(os.path.join(LOCAL_AUDIO,stem+'*.wav')))
    if not cands: continue
    gt=load_notes_gt(jp); pred=base_inference(cands[0])
    P,R,F=match_notes(gt,pred)
    rows.append({'id':stem,'P50':P,'R50':R,'F50':F,'n_gt':len(gt)})
base_df=pd.DataFrame(rows)
base_agg={k:float(np.average(base_df[k],weights=base_df.n_gt)) for k in ['P50','R50','F50']}
print(f"Pretrained BP -> P50:{base_agg['P50']:.4f}  R50:{base_agg['R50']:.4f}  F50:{base_agg['F50']:.4f}")

Evaluating pretrained Basic Pitch on test set (player 05)...
Pretrained BP -> P50:0.6086  R50:0.9018  F50:0.7115


In [30]:
ft_df, ft_agg = evaluate(model, LOCAL_JAMS, LOCAL_AUDIO, "Fine-tuned Basic Pitch (Acoustic)")

print(f"\n{'':30s} {'P50':>7} {'R50':>7} {'F50':>7}")
print(f"{'Pretrained Basic Pitch':30s} {base_agg['P50']:>7.4f} {base_agg['R50']:>7.4f} {base_agg['F50']:>7.4f}")
print(f"{'Fine-tuned (Acoustic)':30s} {ft_agg['P50']:>7.4f} {ft_agg['R50']:>7.4f} {ft_agg['F50']:>7.4f}")
print(f"{'Delta':30s} {ft_agg['P50']-base_agg['P50']:>+7.4f} {ft_agg['R50']-base_agg['R50']:>+7.4f} {ft_agg['F50']-base_agg['F50']:>+7.4f}")
print(f"\nNote: existing pipeline BP F1 = 0.7758 (full audio, threshold 0.40/0.30)")


── Fine-tuned Basic Pitch (Acoustic) ──
  P50:0.6154  R50:0.4762  F50:0.5189  n=60

                                   P50     R50     F50
Pretrained Basic Pitch          0.6086  0.9018  0.7115
Fine-tuned (Acoustic)           0.6154  0.4762  0.5189
Delta                          +0.0068 -0.4256 -0.1926

Note: existing pipeline BP F1 = 0.7758 (full audio, threshold 0.40/0.30)


In [31]:
# Check if pretrained weights loaded correctly by comparing predictions
# Pretrained SavedModel output vs Keras model output on same input
import numpy as np, tensorflow as tf, librosa
from basic_pitch import ICASSP_2022_MODEL_PATH
from basic_pitch.constants import AUDIO_SAMPLE_RATE, AUDIO_N_SAMPLES

# Load a test clip
test_jams_list = [j for j in sorted(glob.glob(os.path.join(LOCAL_JAMS,'*.jams')))
                  if os.path.basename(j).split('_')[0]=='05']
stem = os.path.splitext(os.path.basename(test_jams_list[0]))[0]
cands = glob.glob(os.path.join(LOCAL_AUDIO, stem+'*mic*.wav'))
y, _ = librosa.load(cands[0], sr=AUDIO_SAMPLE_RATE, mono=True)
chunk = y[:AUDIO_N_SAMPLES]
if len(chunk)<AUDIO_N_SAMPLES: chunk=np.pad(chunk,(0,AUDIO_N_SAMPLES-len(chunk)))
x = tf.constant(chunk.reshape(1,-1,1), dtype=tf.float32)

# SavedModel prediction
saved_model = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))
out_saved = saved_model.signatures['serving_default'](input_2=x)

# Keras model prediction (should match if weights loaded correctly)
out_keras = model(x, training=False)

print("SavedModel onset max:", out_saved['onset'].numpy().max())
print("Keras model onset max:", out_keras['onset'].numpy().max())
print("Are they close?", np.allclose(out_saved['onset'].numpy(), out_keras['onset'].numpy(), atol=1e-3))
print("\nSavedModel note mean:", out_saved['note'].numpy().mean())
print("Keras note mean:", out_keras['note'].numpy().mean())

SavedModel onset max: 0.9307201
Keras model onset max: 0.31619126
Are they close? False

SavedModel note mean: 0.117839225
Keras note mean: 0.1172598


In [32]:
# Match weights by name instead of by position
saved_model = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))

# See what names exist in both
saved_vars = {v.name: v for v in saved_model.variables}
keras_vars = {v.name: v for v in model.variables}

print("SavedModel variable names:")
for name in sorted(saved_vars.keys())[:10]:
    print(f"  {name}: {saved_vars[name].shape}")

print("\nKeras model variable names:")
for name in sorted(keras_vars.keys())[:10]:
    print(f"  {name}: {keras_vars[name].shape}")

SavedModel variable names:
  batch_normalization/beta:0: (1,)
  batch_normalization/gamma:0: (1,)
  batch_normalization/moving_mean:0: (1,)
  batch_normalization/moving_variance:0: (1,)
  batch_normalization_2/beta:0: (8,)
  batch_normalization_2/gamma:0: (8,)
  batch_normalization_2/moving_mean:0: (8,)
  batch_normalization_2/moving_variance:0: (8,)
  batch_normalization_3/beta:0: (32,)
  batch_normalization_3/gamma:0: (32,)

Keras model variable names:
  beta: (32,)
  bias: (1,)
  gamma: (32,)
  kernel: (3, 3, 33, 1)
  moving_mean: (32,)
  moving_variance: (32,)


In [33]:
# Match by shape order - both models have same architecture, just different naming
saved_vars_ordered = sorted(saved_model.variables, key=lambda v: v.name)
keras_vars_ordered = sorted(model.variables, key=lambda v: v.name)

print(f"SavedModel: {len(saved_vars_ordered)} vars")
print(f"Keras model: {len(keras_vars_ordered)} vars")

# Print shapes side by side to see if they align
print("\nShape comparison:")
for i, (sv, kv) in enumerate(zip(saved_vars_ordered, keras_vars_ordered)):
    match = "OK" if sv.shape == kv.shape else "MISMATCH"
    print(f"  [{i}] saved={sv.name} {sv.shape} | keras={kv.name} {kv.shape} {match}")

SavedModel: 24 vars
Keras model: 24 vars

Shape comparison:
  [0] saved=batch_normalization/beta:0 (1,) | keras=beta (1,) OK
  [1] saved=batch_normalization/gamma:0 (1,) | keras=beta (8,) MISMATCH
  [2] saved=batch_normalization/moving_mean:0 (1,) | keras=beta (32,) MISMATCH
  [3] saved=batch_normalization/moving_variance:0 (1,) | keras=bias (8,) MISMATCH
  [4] saved=batch_normalization_2/beta:0 (8,) | keras=bias (1,) MISMATCH
  [5] saved=batch_normalization_2/gamma:0 (8,) | keras=bias (32,) MISMATCH
  [6] saved=batch_normalization_2/moving_mean:0 (8,) | keras=bias (32,) MISMATCH
  [7] saved=batch_normalization_2/moving_variance:0 (8,) | keras=bias (1,) MISMATCH
  [8] saved=batch_normalization_3/beta:0 (32,) | keras=bias (1,) MISMATCH
  [9] saved=batch_normalization_3/gamma:0 (32,) | keras=gamma (1,) MISMATCH
  [10] saved=batch_normalization_3/moving_mean:0 (32,) | keras=gamma (8,) MISMATCH
  [11] saved=batch_normalization_3/moving_variance:0 (32,) | keras=gamma (32,) OK
  [12] saved=c

In [34]:
# Sort both by (shape, variable type) — kernels before biases/betas
def sort_key(v):
    name = v.name if hasattr(v, 'name') else v.path
    shape_str = str(list(v.shape))
    # kernel > bias > gamma > beta > moving_mean > moving_variance
    order = {'kernel':0,'bias':1,'gamma':2,'beta':3,'moving_mean':4,'moving_variance':5}
    param = name.split('/')[-1].replace(':0','').split('_')[0]
    return (shape_str, order.get(param, 9))

saved_sorted = sorted(saved_model.variables, key=sort_key)
keras_sorted = sorted(model.variables, key=sort_key)

print("Sorted shape comparison:")
all_match = True
for i, (sv, kv) in enumerate(zip(saved_sorted, keras_sorted)):
    match = sv.shape == kv.shape
    if not match: all_match = False
    sname = sv.name.split('/')[-1].replace(':0','')
    kname = kv.path.split('/')[-1] if '/' in kv.path else kv.path
    print(f"  [{i}] {sname} {sv.shape} -> {kname} {kv.shape} {'OK' if match else 'MISMATCH'}")

if all_match:
    print("\nAll shapes match! Transferring weights...")
    for sv, kv in zip(saved_sorted, keras_sorted):
        kv.assign(sv)
    print("Done. Verifying...")
    out_saved = saved_model.signatures['serving_default'](input_2=x)
    out_keras = model(x, training=False)
    print(f"SavedModel onset max: {out_saved['onset'].numpy().max():.4f}")
    print(f"Keras onset max: {out_keras['onset'].numpy().max():.4f}")
    print(f"Close: {np.allclose(out_saved['onset'].numpy(), out_keras['onset'].numpy(), atol=0.01)}")

Sorted shape comparison:
  [0] bias (1,) -> bias (1,) OK
  [1] bias (1,) -> bias (1,) OK
  [2] bias (1,) -> bias (1,) OK
  [3] gamma (1,) -> gamma (1,) OK
  [4] beta (1,) -> beta (1,) OK
  [5] moving_mean (1,) -> moving_mean (1,) OK
  [6] moving_variance (1,) -> moving_variance (1,) OK
  [7] kernel (3, 3, 33, 1) -> kernel (3, 3, 33, 1) OK
  [8] kernel (3, 39, 8, 8) -> kernel (3, 39, 8, 8) OK
  [9] bias (32,) -> bias (32,) OK
  [10] bias (32,) -> bias (32,) OK
  [11] gamma (32,) -> gamma (32,) OK
  [12] beta (32,) -> beta (32,) OK
  [13] moving_mean (32,) -> moving_mean (32,) OK
  [14] moving_variance (32,) -> moving_variance (32,) OK
  [15] kernel (5, 5, 8, 1) -> kernel (5, 5, 8, 1) OK
  [16] kernel (5, 5, 8, 32) -> kernel (5, 5, 8, 32) OK
  [17] kernel (7, 3, 32, 1) -> kernel (7, 3, 32, 1) OK
  [18] kernel (7, 7, 1, 32) -> kernel (7, 7, 1, 32) OK
  [19] bias (8,) -> bias (8,) OK
  [20] gamma (8,) -> gamma (8,) OK
  [21] beta (8,) -> beta (8,) OK
  [22] moving_mean (8,) -> moving_mean 

In [35]:
# Recompile with weighted loss (fixes recall collapse)
def weighted_bce(y_true, y_pred):
    pos_weight = 5.0
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.05)
    weight = 1.0 + (pos_weight - 1.0) * tf.reduce_mean(y_true, axis=-1)
    return tf.reduce_mean(weight * bce)

model.compile(
    loss={'onset': weighted_bce, 'note': weighted_bce, 'contour': weighted_bce},
    optimizer=tf.keras.optimizers.Adam(1e-4),
)

OUTPUT_DIR2 = '/content/drive/MyDrive/Capstone/outputs/bp_finetune_v2'
os.makedirs(OUTPUT_DIR2, exist_ok=True)

callbacks2 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR2, 'best_model.keras'),
        save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=10, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
]

history2 = model.fit(
    train_ds_no_weights,
    epochs=50,
    steps_per_epoch=100,
    validation_data=val_ds_no_weights,
    validation_steps=20,
    callbacks=callbacks2,
)

Epoch 1/50
 97/100 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - contour_loss: 0.1744 - loss: 0.4982 - note_loss: 0.1748 - onset_loss: 0.1489
Epoch 1: val_loss improved from None to 0.40290, saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune_v2/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune_v2/best_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - contour_loss: 0.1656 - loss: 0.4630 - note_loss: 0.1627 - onset_loss: 0.1347 - val_contour_loss: 0.1421 - val_loss: 0.4029 - val_note_loss: 0.1402 - val_onset_loss: 0.1207 - learning_rate: 1.0000e-04
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - contour_loss: 0.1479 - loss: 0.4169 - note_loss: 0.1478 - onset_loss: 0.1213
Epoch 2: val_loss improved from 0.40290 to 0.39175, saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune_v2/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Capstone/outputs/bp_finetune_v2/best_model.keras
10

In [36]:
ft_df2, ft_agg2 = evaluate(model, LOCAL_JAMS, LOCAL_AUDIO, "Fine-tuned v2 (weighted loss)")

print(f"\n{'':30s} {'P50':>7} {'R50':>7} {'F50':>7}")
print(f"{'Pretrained Basic Pitch':30s} {base_agg['P50']:>7.4f} {base_agg['R50']:>7.4f} {base_agg['F50']:>7.4f}")
print(f"{'Fine-tuned v2 (weighted)':30s} {ft_agg2['P50']:>7.4f} {ft_agg2['R50']:>7.4f} {ft_agg2['F50']:>7.4f}")
print(f"{'Delta':30s} {ft_agg2['P50']-base_agg['P50']:>+7.4f} {ft_agg2['R50']-base_agg['R50']:>+7.4f} {ft_agg2['F50']-base_agg['F50']:>+7.4f}")


── Fine-tuned v2 (weighted loss) ──
  P50:0.7016  R50:0.5622  F50:0.6029  n=60

                                   P50     R50     F50
Pretrained Basic Pitch          0.6086  0.9018  0.7115
Fine-tuned v2 (weighted)        0.7016  0.5622  0.6029
Delta                          +0.0930 -0.3395 -0.1086


In [37]:
# Try much lower thresholds on the fine-tuned model
for onset_t, frame_t in [(0.3, 0.2), (0.2, 0.1), (0.1, 0.05)]:
    rows = []
    for jp in test_jams_list:
        stem = os.path.splitext(os.path.basename(jp))[0]
        cands = (glob.glob(os.path.join(LOCAL_AUDIO, stem+'*mic*.wav')) or
                 glob.glob(os.path.join(LOCAL_AUDIO, stem+'*.wav')))
        if not cands: continue
        y, _ = librosa.load(cands[0], sr=AUDIO_SAMPLE_RATE, mono=True)
        all_onset=[]; all_note=[]; all_contour=[]
        for start in range(0, len(y), AUDIO_N_SAMPLES):
            chunk = y[start:start+AUDIO_N_SAMPLES]
            if len(chunk)<AUDIO_N_SAMPLES: chunk=np.pad(chunk,(0,AUDIO_N_SAMPLES-len(chunk)))
            x = tf.constant(chunk.reshape(1,-1,1), dtype=tf.float32)
            out = model(x, training=False)
            all_onset.append(out['onset'][0].numpy())
            all_note.append(out['note'][0].numpy())
            all_contour.append(out['contour'][0].numpy())
        onset_mat=np.concatenate(all_onset,0)
        note_mat=np.concatenate(all_note,0)
        contour_mat=np.concatenate(all_contour,0)
        _, note_events = model_output_to_notes(
            {'onset':onset_mat,'note':note_mat,'contour':contour_mat},
            onset_thresh=onset_t, frame_thresh=frame_t,
            min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
            min_freq=None, max_freq=None, include_pitch_bends=False)
        pred = [{'onset':float(n[0]),'offset':float(n[1]),'midi':int(n[2])} for n in note_events]
        gt = load_notes_gt(jp)
        P,R,F = match_notes(gt, pred)
        rows.append({'P50':P,'R50':R,'F50':F,'n_gt':len(gt)})
    df = pd.DataFrame(rows)
    agg = {k: float(np.average(df[k],weights=df.n_gt)) for k in ['P50','R50','F50']}
    print(f"onset={onset_t} frame={frame_t} -> P50:{agg['P50']:.4f} R50:{agg['R50']:.4f} F50:{agg['F50']:.4f}")

onset=0.3 frame=0.2 -> P50:0.6548 R50:0.6165 F50:0.6140
onset=0.2 frame=0.1 -> P50:0.5538 R50:0.7407 F50:0.6191
onset=0.1 frame=0.05 -> P50:0.3655 R50:0.9061 F50:0.5107


In [39]:
import numpy as np, pandas as pd, librosa, tensorflow as tf
from basic_pitch.note_creation import model_output_to_notes
from basic_pitch.constants import AUDIO_SAMPLE_RATE, AUDIO_N_SAMPLES, ANNOTATIONS_FPS

# Step 1: cache all model outputs (runs once, ~2 min)
print("Caching model outputs for test set...")
cached = {}
for jp in test_jams_list:
    stem = os.path.splitext(os.path.basename(jp))[0]
    cands = (glob.glob(os.path.join(LOCAL_AUDIO, stem+'*mic*.wav')) or
             glob.glob(os.path.join(LOCAL_AUDIO, stem+'*.wav')))
    if not cands: continue
    y, _ = librosa.load(cands[0], sr=AUDIO_SAMPLE_RATE, mono=True)
    all_onset=[]; all_note=[]; all_contour=[]
    for start in range(0, len(y), AUDIO_N_SAMPLES):
        chunk = y[start:start+AUDIO_N_SAMPLES]
        if len(chunk)<AUDIO_N_SAMPLES: chunk=np.pad(chunk,(0,AUDIO_N_SAMPLES-len(chunk)))
        x = tf.constant(chunk.reshape(1,-1,1), dtype=tf.float32)
        out = model(x, training=False)
        all_onset.append(out['onset'][0].numpy())
        all_note.append(out['note'][0].numpy())
        all_contour.append(out['contour'][0].numpy())
    cached[stem] = {
        'onset': np.concatenate(all_onset, 0),
        'note':  np.concatenate(all_note,  0),
        'contour': np.concatenate(all_contour, 0),
        'gt': load_notes_gt(jp)
    }
print(f"Cached {len(cached)} recordings")

# Step 2: sweep thresholds instantly
import itertools
results = []
for onset_t, frame_t in itertools.product(
    [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40],
    [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
):
    rows = []
    for stem, data in cached.items():
        _, note_events = model_output_to_notes(
            {'onset':data['onset'],'note':data['note'],'contour':data['contour']},
            onset_thresh=onset_t, frame_thresh=frame_t,
            min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
            min_freq=None, max_freq=None, include_pitch_bends=False)
        pred = [{'onset':float(n[0]),'offset':float(n[1]),'midi':int(n[2])} for n in note_events]
        P,R,F = match_notes(data['gt'], pred)
        rows.append({'P50':P,'R50':R,'F50':F,'n_gt':len(data['gt'])})
    df = pd.DataFrame(rows)
    agg = {k: float(np.average(df[k],weights=df.n_gt)) for k in ['P50','R50','F50']}
    results.append({'onset_t':onset_t,'frame_t':frame_t,**agg})

results_df = pd.DataFrame(results).sort_values('F50', ascending=False)
print(results_df.head(10).to_string(index=False))
print(f"\nPretrained baseline: P50=0.6086 R50=0.9018 F50=0.7115 (onset=0.40, frame=0.30)")

Caching model outputs for test set...
Cached 60 recordings
 onset_t  frame_t      P50      R50      F50
    0.10     0.30 0.704205 0.890189 0.775927
    0.15     0.30 0.743981 0.829948 0.773936
    0.10     0.25 0.684083 0.893287 0.764404
    0.15     0.25 0.722458 0.833620 0.763257
    0.10     0.20 0.660573 0.899025 0.751097
    0.15     0.20 0.695519 0.837292 0.748947
    0.20     0.30 0.760355 0.756627 0.744316
    0.20     0.25 0.734582 0.757085 0.731203
    0.10     0.15 0.619787 0.903270 0.724713
    0.15     0.15 0.650068 0.838210 0.721149

Pretrained baseline: P50=0.6086 R50=0.9018 F50=0.7115 (onset=0.40, frame=0.30)


In [38]:
import itertools

results = []
for onset_t, frame_t in itertools.product(
    [0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    [0.05, 0.10, 0.15, 0.20, 0.25]
):
    rows = []
    for jp in test_jams_list:
        stem = os.path.splitext(os.path.basename(jp))[0]
        cands = (glob.glob(os.path.join(LOCAL_AUDIO, stem+'*mic*.wav')) or
                 glob.glob(os.path.join(LOCAL_AUDIO, stem+'*.wav')))
        if not cands: continue
        y, _ = librosa.load(cands[0], sr=AUDIO_SAMPLE_RATE, mono=True)
        all_onset=[]; all_note=[]; all_contour=[]
        for start in range(0, len(y), AUDIO_N_SAMPLES):
            chunk = y[start:start+AUDIO_N_SAMPLES]
            if len(chunk)<AUDIO_N_SAMPLES: chunk=np.pad(chunk,(0,AUDIO_N_SAMPLES-len(chunk)))
            x = tf.constant(chunk.reshape(1,-1,1), dtype=tf.float32)
            out = model(x, training=False)
            all_onset.append(out['onset'][0].numpy())
            all_note.append(out['note'][0].numpy())
            all_contour.append(out['contour'][0].numpy())
        onset_mat=np.concatenate(all_onset,0)
        note_mat=np.concatenate(all_note,0)
        contour_mat=np.concatenate(all_contour,0)
        _, note_events = model_output_to_notes(
            {'onset':onset_mat,'note':note_mat,'contour':contour_mat},
            onset_thresh=onset_t, frame_thresh=frame_t,
            min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
            min_freq=None, max_freq=None, include_pitch_bends=False)
        pred = [{'onset':float(n[0]),'offset':float(n[1]),'midi':int(n[2])} for n in note_events]
        gt = load_notes_gt(jp)
        P,R,F = match_notes(gt, pred)
        rows.append({'P50':P,'R50':R,'F50':F,'n_gt':len(gt)})
    df = pd.DataFrame(rows)
    agg = {k: float(np.average(df[k],weights=df.n_gt)) for k in ['P50','R50','F50']}
    results.append({'onset_t':onset_t,'frame_t':frame_t,**agg})

results_df = pd.DataFrame(results).sort_values('F50', ascending=False)
print(results_df.head(10).to_string(index=False))
print(f"\nBest F50: {results_df.iloc[0]['F50']:.4f} at onset={results_df.iloc[0]['onset_t']} frame={results_df.iloc[0]['frame_t']}")
print(f"\nPretrained baseline: F50=0.7115 (onset=0.40, frame=0.30)")

KeyboardInterrupt: 

In [26]:
# Check what note_events actually contains
result = model_output_to_notes(
    {'onset':onset_mat,'note':note_mat,'contour':contour_mat},
    onset_thresh=ONSET_THRESHOLD, frame_thresh=FRAME_THRESHOLD,
    min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
    min_freq=None, max_freq=None, include_pitch_bends=False)
print(type(result))
if isinstance(result, tuple):
    print(f"tuple len: {len(result)}")
    for i, item in enumerate(result):
        print(f"  [{i}] type={type(item)}")
        if hasattr(item, '__len__'): print(f"       len={len(item)}")
        if hasattr(item, '__iter__'):
            first = next(iter(item), None)
            if first: print(f"       first item type={type(first)}, value={first}")

NameError: name 'onset_mat' is not defined

In [27]:
import numpy as np
from basic_pitch.note_creation import model_output_to_notes
from basic_pitch.constants import ANNOTATIONS_FPS

# Fake tiny inputs to inspect return format
fake = {'onset': np.zeros((10,88)), 'note': np.zeros((10,88)), 'contour': np.zeros((10,264))}
result = model_output_to_notes(
    fake,
    onset_thresh=0.4, frame_thresh=0.3,
    min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
    min_freq=None, max_freq=None, include_pitch_bends=False)

print(type(result))
if isinstance(result, tuple):
    for i, item in enumerate(result):
        print(f"[{i}] type={type(item)}, repr={repr(item)[:100]}")

<class 'tuple'>
[0] type=<class 'pretty_midi.pretty_midi.PrettyMIDI'>, repr=<pretty_midi.pretty_midi.PrettyMIDI object at 0x788960e0f2f0>
[1] type=<class 'list'>, repr=[]


In [28]:
# Check note object format with real audio
import librosa
from basic_pitch.constants import AUDIO_SAMPLE_RATE, AUDIO_N_SAMPLES, ANNOTATIONS_FPS
import tensorflow as tf, numpy as np

test_jams_list = [j for j in sorted(glob.glob(os.path.join(LOCAL_JAMS,'*.jams')))
                  if os.path.basename(j).split('_')[0]=='05']
y, _ = librosa.load(
    glob.glob(os.path.join(LOCAL_AUDIO,
              os.path.splitext(os.path.basename(test_jams_list[0]))[0]+'*mic*.wav'))[0],
    sr=AUDIO_SAMPLE_RATE, mono=True)
chunk = y[:AUDIO_N_SAMPLES]
if len(chunk)<AUDIO_N_SAMPLES: chunk=np.pad(chunk,(0,AUDIO_N_SAMPLES-len(chunk)))
x = tf.constant(chunk.reshape(1,-1,1), dtype=tf.float32)
out = base_model.signatures['serving_default'](input_2=x)
result = model_output_to_notes(
    {'onset':out['onset'][0].numpy(),'note':out['note'][0].numpy(),'contour':out['contour'][0].numpy()},
    onset_thresh=0.4, frame_thresh=0.3,
    min_note_len=int(np.round(58/1000*ANNOTATIONS_FPS)),
    min_freq=None, max_freq=None, include_pitch_bends=False)
midi_obj, note_events = result
print(f"note_events type: {type(note_events)}, len: {len(note_events)}")
if note_events:
    print(f"first note: {note_events[0]}")
    print(f"first note type: {type(note_events[0])}")
    print(f"first note attrs: {dir(note_events[0])}")

note_events type: <class 'list'>, len: 20
first note: (np.float64(1.8575963718820863), np.float64(1.9853061224489796), np.int64(51), np.float32(0.71119374), None)
first note type: <class 'tuple'>
first note attrs: ['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__rmul__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'count', 'index']


In [ ]:
# ── Fine-tuned model eval ────────────────────────────────────────────────────
print("Loading fine-tuned model...")
ft_model = tf.keras.models.load_model(os.path.join(OUTPUT_DIR, 'best_model'))
ft_df, ft_agg = evaluate(ft_model, LOCAL_JAMS, LOCAL_AUDIO, "Fine-tuned Basic Pitch (Acoustic)")

print(f"\n{'':30s} {'P50':>7} {'R50':>7} {'F50':>7}")
print(f"{'Basic Pitch pretrained':30s} {base_agg['P50']:>7.4f} {base_agg['R50']:>7.4f} {base_agg['F50']:>7.4f}")
print(f"{'Fine-tuned (Acoustic)':30s}  {ft_agg['P50']:>7.4f}  {ft_agg['R50']:>7.4f}  {ft_agg['F50']:>7.4f}")
print(f"{'Delta':30s} {ft_agg['P50']-base_agg['P50']:>+7.4f} {ft_agg['R50']-base_agg['R50']:>+7.4f} {ft_agg['F50']-base_agg['F50']:>+7.4f}")
print(f"\nNote: existing pipeline Basic Pitch F1 = 0.7758 (different eval — Basic Pitch predict() API at threshold 0.40)")